<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# HotelChain West Analytics
## Notebook 2 — Dashboard Power BI & Storytelling

> **Prérequis** : Notebook 1 complété. Les 4 fichiers CSV exportés doivent être disponibles dans `SAVE_PATH`.
>
> - **Colab** : `/content/drive/MyDrive/DataProjectLab/projects/hotelchain_analytics/`
> - **Local** : `./outputs/`

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Power BI Desktop |
| **Durée estimée** | 3h à 4h |

> 💡 **Ce notebook est un guide de conception.** Tu vas travailler directement dans Power BI Desktop en suivant les instructions ci-dessous.

---
## 1. Sources de données à importer dans Power BI

Les fichiers produits par le Notebook 1 à importer dans Power BI :

| Fichier | Contenu |
|---|---|
| `hotelchain_occupation.csv` | Taux d'occupation mensuel par hôtel |
| `hotelchain_revpar.csv` | ADR et RevPAR mensuel par hôtel |
| `hotelchain_clients_clv.csv` | Segmentation clients (quartile, VIP) |
| `hotelchain_canaux.csv` | Performance par canal de réservation |

Plus les fichiers bruts depuis GitHub :

| Fichier | Contenu |
|---|---|
| `hotels.csv` | Référentiel hôtels |
| `services.csv` | Revenus extras par réservation |

### Comment importer
1. Power BI Desktop → **Obtenir des données → Texte/CSV**
2. Sélectionner chaque fichier
3. Vérifier les types de colonnes dans Power Query
4. Choisir le mode **Import**

---
## 2. Modèle de données — Schéma en étoile

```
                    hotels (dimension)
                        |
hotelchain_canaux ──────┤
                        |
hotelchain_clients ─── hotelchain_occupation (FAIT) ─── hotelchain_revpar
                        |
                    services
                        |
                    Calendrier (dimension temps)
```

### Table Calendrier — à créer dans Power BI

**Nouvelle table → coller ce code DAX :**

```dax
Calendrier =
ADDCOLUMNS(
    CALENDAR(DATE(2022,1,1), DATE(2024,12,31)),
    "Annee",       YEAR([Date]),
    "Mois_Num",    MONTH([Date]),
    "Mois_Nom",    FORMAT([Date], "MMM YYYY"),
    "Trimestre",   "T" & QUARTER([Date]),
    "Semaine",     WEEKNUM([Date]),
    "Jour_Semaine", FORMAT([Date], "ddd"),
    "Est_Weekend",  IF(WEEKDAY([Date],2) >= 6, 1, 0),
    "Saison",      IF(MONTH([Date]) IN {12,1,2,7,8}, "Haute saison", "Basse saison")
)
```

### Relations à créer

| Table source | Clé | Table cible | Clé |
|---|---|---|---|
| `hotelchain_occupation` | `nom` | `hotels` | `nom` |
| `hotelchain_revpar` | `nom` | `hotels` | `nom` |
| `hotelchain_occupation` | `annee` + `mois` | `Calendrier` | `Annee` + `Mois_Num` |
| `hotelchain_clients_clv` | — | `hotels` | — |

> ⚠️ Toutes les relations en **One-to-Many**, direction de filtre **Single (→)**.

---
## 3. Structure du dashboard — 5 pages

| Page | Question business principale |
|---|---|
| 1. Vue executive | Quelle est la santé globale de la chaîne ? |
| 2. Occupation & Saisonnalité | Quels hôtels et quelles périodes performent ? |
| 3. Revenus & Tarification | Quels canaux et quels prix maximisent le RevPAR ? |
| 4. Clients & Satisfaction | Qui génère le plus de valeur ? |
| 5. Services & Extras | Quel est l'impact des upsells sur la satisfaction ? |

---
## 4. Page 1 — Vue executive

**Question business :** Quelle est la santé globale de la chaîne ?

### KPI Cards (ligne 1)
Revenu Total · Taux Occupation moyen · CSAT Moyen · Taux Annulation

### Visuels (lignes 2 et 3)
- **Courbe** : évolution mensuelle du RevPAR par hôtel
- **Barres horizontales** : top 5 canaux par revenu
- **Carte géographique** : revenu par pays (Côte d'Ivoire, Sénégal, Cameroun, Ghana)
- **Slicers** : Année, Hôtel, Saison

### Colonnes utiles
`revenu_total`, `taux_occupation_pct`, `csat_moyen`, `taux_annulation_pct`, `revpar`, `canal`, `annee`, `mois`

---
## 5. Page 2 — Occupation & Saisonnalité

**Question business :** Quels hôtels et quelles périodes ont les meilleurs taux d'occupation ?

### Visuels
- **Matrice (heatmap)** : taux occupation par hôtel (lignes) × mois (colonnes)
  - Mise en forme conditionnelle : 🔴 < 40%, 🟠 40-60%, 🟢 > 60%
- **Courbe** : taux occupation mensuel avec ligne cible à 65%
- **Barres groupées** : haute saison vs basse saison par hôtel
- **KPI cards** : meilleur mois, pire mois

### Colonnes utiles
`nom`, `annee`, `mois`, `taux_occupation_pct`, `nuits_vendues`, `rang_mois`

---
## 6. Page 3 — Revenus & Tarification

**Question business :** Quels canaux et quels prix maximisent le RevPAR ?

### Visuels
- **Barres** : ADR par hôtel avec comparaison mensuelle
- **Aires empilées** : revenus mensuels par canal
- **Scatter** : ADR × Taux occupation par hôtel (identifier le sweet spot tarifaire)
- **Tableau** : synthèse RevPAR par hôtel et par trimestre avec `rang_revpar`

### Colonnes utiles
`nom`, `adr`, `revpar`, `taux_occupation_pct`, `annee`, `mois`, `rang_revpar`, `canal`, `revenu_total`, `variation`

---
## 7. Page 4 — Clients & Satisfaction

**Question business :** Qui génère le plus de valeur ?

### Visuels
- **Tableau** : segmentation clients (quartile, revenu, nb séjours, CSAT, VIP)
- **Barres** : CSAT moyen par hôtel avec cible 4.0
- **Anneau** : répartition nationalités top 5
- **KPI** : % clients VIP, revenu moyen par quartile, écart fidèles vs nouveaux

### Colonnes utiles
`client_nom`, `nationalite`, `client_fidele`, `nb_sejours`, `revenu_total`, `csat_moyen`, `quartile_valeur`, `segment_vip`

---
## 8. Page 5 — Services & Extras

**Question business :** Quel est l'impact des upsells sur le revenu et la satisfaction ?

### Visuels
- **Barres** : revenu extras par catégorie (Restaurant, Spa, Minibar...)
- **Scatter** : corrélation revenu extras × CSAT par réservation
- **Courbe** : évolution mensuelle du revenu extras
- **KPI** : revenu extras / revenu total (% upsell)

### Colonnes utiles
`categorie_service`, `montant`, `reservation_id`, `csat`

---
## 9. Les 12 mesures DAX

Créer toutes les mesures dans une table vide `_Mesures` :

### Mesures de base

```dax
Total Reservations =
COUNTROWS(hotelchain_occupation)

Revenu Total =
SUM(hotelchain_revpar[revenu])

CSAT Moyen =
AVERAGE(hotelchain_clients_clv[csat_moyen])

Taux Annulation =
AVERAGE(hotelchain_occupation[taux_annulation_pct]) / 100
```

### Mesures avancées

```dax
Nuits Vendues =
SUM(hotelchain_occupation[nuits_vendues])

Taux Occupation =
AVERAGE(hotelchain_occupation[taux_occupation_pct]) / 100

ADR =
AVERAGE(hotelchain_revpar[adr])

RevPAR =
AVERAGE(hotelchain_revpar[revpar])

Revenu Mois Prec =
CALCULATE([Revenu Total], PREVIOUSMONTH(Calendrier[Date]))

Variation Revenu % =
DIVIDE([Revenu Total] - [Revenu Mois Prec], [Revenu Mois Prec])

Revenu Extras =
SUM(services[montant])

Revenu Total Consolidé =
[Revenu Total] + [Revenu Extras]
```

### Mesures dynamiques

```dax
Sous Titre Page 1 =
VAR _hotel  = SELECTEDVALUE(hotels[nom], "Tous les hôtels")
VAR _annee  = SELECTEDVALUE(Calendrier[Annee], "2022-2024")
VAR _saison = SELECTEDVALUE(Calendrier[Saison], "Toutes saisons")
RETURN
"HotelChain West  ·  " & _hotel & "  ·  " & _annee & "  ·  " & _saison

Pct Extras =
DIVIDE([Revenu Extras], [Revenu Total Consolidé])
```

---
## 10. Checklist de validation

### Import & Modèle
- [ ] 6 fichiers importés sans erreur
- [ ] Table Calendrier créée et marquée comme table de dates
- [ ] Toutes les relations définies (Single, One-to-Many)

### Mesures DAX
- [ ] 12 mesures créées dans `_Mesures`
- [ ] Valeurs cohérentes avec les résultats du Notebook 1

### Dashboard
- [ ] Page 1 — Vue executive avec 4 KPIs
- [ ] Page 2 — Heatmap occupation avec mise en forme conditionnelle
- [ ] Page 3 — Scatter ADR × Taux occupation
- [ ] Page 4 — Tableau segmentation clients avec colonnes VIP et quartile
- [ ] Page 5 — Corrélation extras × CSAT
- [ ] Slicers globaux fonctionnels sur toutes les pages

### Valeurs attendues (sans filtre)

| Mesure DAX | Valeur |
|---|---|
| [Revenu Total] | ? FCFA |
| [CSAT Moyen] | ~4.3 / 5 |
| [Taux Annulation] | ~4% |
| [RevPAR] meilleur hôtel | ? |
| [Pct Extras] | ? % |

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.